# Final Evaluation - Test Set Opened Once

**Razorpay AI Buildathon 2026 - Track 02: AI Risk Manager**

Return-Risk Scorer for COD orders (Return-to-Origin prediction).

---

### Protocol

Everything that constitutes a **choice** happens on validation, before the test set is
touched:

| Chosen on validation | Section |
|---|---|
| Which model ships | 2 |
| The operating threshold (rupee cost minimum) | 2 |
| The three-tier cut points | 2 |

Then the test set is opened **once**, in section 3, and every number after that is
computed at those frozen settings. Re-tuning anything after section 3 would make every
figure below an in-sample figure, and the honest thing would be to say so — which is
what `WHAT_BROKE.md` is for.

### What gets reported, per `PRE_REGISTRATION.md`

- precision, recall, F1, PR-AUC, ROC-AUC, confusion matrix at the cost-optimal threshold
- **calibration** — Brier score and a reliability diagram
- **cost in rupees**, with the false-negative cost sensitivity-tested across ₹150–250
- what the default 0.5 threshold would have cost instead
- the three-tier policy: allow / charge a COD fee / disable COD
- the **prevalence-shift study** across 18–35%, the real range across Indian cities
- SHAP explanations rendered as human-readable reasons
- comparison across all 11 benchmark entries plus the tuned finalists
- an **honest exception list** — where this model underperforms, named and quantified

## 1. Load tuned finalists

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.costs import (
    ALLOW, BLOCK, FEE, TIERS,
    CostParams, apply_tiers, binary_cost, no_model_baselines,
    optimal_threshold, optimise_tiers, sensitivity, three_tier_cost, threshold_sweep,
)
from src.evaluate import (
    classification_metrics, paired_bootstrap_delta, prevalence_curve,
    prior_shift_correction, reliability_curve, resample_to_prevalence,
    risk_reasons, segment_report,
)
from src.features import TARGET
from src.generate import load_evidence
from src.models import SEED, model_input_columns, model_zoo, to_design_matrix

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

FIG = ROOT / "reports/figures"
RES = ROOT / "reports/results"
MODELS = ROOT / "models"

COLS = model_input_columns()
zoo = model_zoo(seed=SEED)
evidence = load_evidence(ROOT / "config/evidence.yaml")
COSTS = CostParams.from_evidence(evidence)

print("Cost assumptions (all marked `assumed` in config/evidence.yaml):")
for f, v in COSTS.__dict__.items():
    print(f"  {f:<22} {v}")
print("\nFN = flat shipping burned. FP = order_value x margin x (1 - prepaid_conversion).")
print("fn_cost_inr and margin_rate are sensitivity-tested in section 8.")

In [ ]:
val = pd.read_parquet(ROOT / "data/processed/val.parquet")
y_val = val[TARGET].to_numpy()

bench = json.load(open(RES / "03_benchmark_summary.json", encoding="utf-8"))
tune = json.load(open(RES / "04_tuning_summary.json", encoding="utf-8"))
FINALISTS = tune["finalists"]

val_pred_untuned = pd.read_parquet(RES / "03_val_predictions.parquet")
val_pred_tuned = pd.read_parquet(RES / "04_tuned_val_predictions.parquet")

tuned_pipes = {p.stem.removeprefix("04_"): joblib.load(p)
               for p in sorted(MODELS.glob("04_*.joblib"))}
print(f"loaded {len(tuned_pipes)} tuned pipelines:")
for k in tuned_pipes:
    print(f"  {k}")

candidates = pd.DataFrame(tune["tuned_val_scores"])
untuned = pd.DataFrame(bench["leaderboard"])
untuned = untuned[untuned.key != "00_dummy"]
print(f"\ncandidate pool: {len(candidates)} tuned + {len(untuned)} untuned")

## 2. Select ONE final model on validation only

In [ ]:
# Every candidate scored on validation. PR-AUC is the pre-registered selection
# metric; Brier is a hard secondary requirement here, because the operating point
# is a rupee cost minimum and a cost curve on badly calibrated scores is fiction.
pool = pd.concat([
    candidates.assign(source="tuned")[
        ["key", "model", "strategy", "val_pr_auc", "val_roc_auc", "val_brier", "source"]],
    untuned.assign(source="untuned", strategy="default")[
        ["key", "model", "strategy", "val_pr_auc", "val_roc_auc", "val_brier", "source"]],
], ignore_index=True).sort_values("val_pr_auc", ascending=False).reset_index(drop=True)

display(pool.round(4).head(12).style.background_gradient(
    subset=["val_pr_auc"], cmap="Greens").hide(axis="index"))

leader_key = pool.key.iat[0]
ALL_VAL_PRED = {**{k: val_pred_untuned[k].to_numpy() for k in val_pred_untuned.columns
                   if k not in ("order_id", "y_true")},
                **{k: val_pred_tuned[k].to_numpy() for k in val_pred_tuned.columns
                   if k not in ("order_id", "y_true")}}

In [ ]:
# Is the leaderboard order real? Paired bootstrap of every candidate against the
# leader. Anything whose CI spans zero is a tie, and a tie is broken on
# calibration and simplicity -- not on the third decimal of PR-AUC.
rows = []
for k in pool.key:
    if k == leader_key:
        continue
    d = paired_bootstrap_delta(y_val, ALL_VAL_PRED[k], ALL_VAL_PRED[leader_key], seed=SEED)
    r = pool[pool.key == k].iloc[0]
    rows.append({"key": k, "model": r.model, "strategy": r.strategy,
                 "val_pr_auc": r.val_pr_auc, "val_brier": r.val_brier,
                 "delta_vs_leader": d["observed_delta"],
                 "ci_lo": d["ci_lo"], "ci_hi": d["ci_hi"],
                 "tied_with_leader": d["ci_lo"] < 0 < d["ci_hi"]})
ties = pd.DataFrame(rows)
display(ties.round(4).head(10).style.hide(axis="index"))

tied_keys = [leader_key] + ties[ties.tied_with_leader].key.tolist()
print(f"\nstatistically tied with the leader: {len(tied_keys)} candidates")

In [ ]:
# SELECTION RULE, applied as written:
#   1. take everything statistically tied with the PR-AUC leader
#   2. among those, take the best Brier score -- calibration decides, because the
#      threshold is a cost minimum
#   3. break any remaining tie on fit cost, because this scores at checkout latency
tied = pool[pool.key.isin(tied_keys)].copy()
tied = tied.sort_values(["val_brier", "val_pr_auc"], ascending=[True, False])
display(tied.round(4).style.hide(axis="index"))

FINAL_KEY = tied.key.iat[0]
FINAL_ROW = pool[pool.key == FINAL_KEY].iloc[0]
FINAL_PIPE = (tuned_pipes[FINAL_KEY] if FINAL_KEY in tuned_pipes else None)

if FINAL_PIPE is None:
    # An untuned entry won: refit it from the zoo on train, once.
    from src.models import build_pipeline
    train_ = pd.read_parquet(ROOT / "data/processed/train.parquet")
    FINAL_PIPE = build_pipeline(zoo[FINAL_KEY])
    FINAL_PIPE.fit(train_[COLS], train_[TARGET].to_numpy())
    del train_

print("=" * 74)
print("FINAL MODEL SELECTED -- on validation only")
print("=" * 74)
print(f"  {FINAL_ROW.model}  ({FINAL_ROW.strategy}, {FINAL_ROW.source})")
print(f"  key            {FINAL_KEY}")
print(f"  val PR-AUC     {FINAL_ROW.val_pr_auc:.4f}")
print(f"  val ROC-AUC    {FINAL_ROW.val_roc_auc:.4f}")
print(f"  val Brier      {FINAL_ROW.val_brier:.4f}   <- why it won its tie group")

In [ ]:
# The operating point, also chosen on validation. Never 0.5, never max-F1.
p_val = ALL_VAL_PRED[FINAL_KEY]
val_values = val.order_value.to_numpy()

THRESHOLD, val_sweep = optimal_threshold(y_val, p_val, val_values, COSTS)
LOW_CUT, HIGH_CUT, tier_surface = optimise_tiers(y_val, p_val, val_values, COSTS)

val_at_thr = binary_cost(y_val, p_val, THRESHOLD, val_values, COSTS)
val_at_half = binary_cost(y_val, p_val, 0.50, val_values, COSTS)

print("FROZEN OPERATING POINT (derived on validation, applied unchanged to test)\n")
print(f"  binary threshold   {THRESHOLD:.3f}")
print(f"  tier cut points    allow < {LOW_CUT:.3f} <= COD fee < {HIGH_CUT:.3f} <= disable COD")
print()
print(f"  on validation, at {THRESHOLD:.3f}: cost/order Rs{val_at_thr['cost_per_order_inr']:.2f}"
      f"   precision {val_at_thr['precision']:.3f}   recall {val_at_thr['recall']:.3f}")
print(f"  on validation, at 0.500: cost/order Rs{val_at_half['cost_per_order_inr']:.2f}"
      f"   precision {val_at_half['precision']:.3f}   recall {val_at_half['recall']:.3f}")

FROZEN = {"model_key": FINAL_KEY, "threshold": THRESHOLD,
          "low_cut": LOW_CUT, "high_cut": HIGH_CUT}
print(f"\nFROZEN = {FROZEN}")
print("\nNothing below this line may change these three numbers.")

## 3. === OPEN TEST SET (first and only time) ===

In [ ]:
print("=" * 74)
print("OPENING THE TEST SET")
print("=" * 74)
print("""
The model, the threshold and the tier cut points are frozen above, chosen on
validation alone. This is the first read of the test split in the entire
project, and it is the only one.
""")

test = pd.read_parquet(ROOT / "data/processed/test.parquet")
y_test = test[TARGET].to_numpy()
test_values = test.order_value.to_numpy()
p_test = FINAL_PIPE.predict_proba(test[COLS])[:, 1]

print(f"test orders   {len(test):,}")
print(f"window        {test.order_ts.min():%Y-%m-%d} -> {test.order_ts.max():%Y-%m-%d}")
print(f"base rate     {y_test.mean():.4f}   (val was {y_val.mean():.4f}, train 0.1696)")
print(f"COD share     {test.is_cod.mean():.4f}")
print()
print("Note the base rate has drifted DOWN across the splits. That is real: the")
print("festive windows sit inside the train period. Section 10 measures what that")
print("kind of drift does to this model, across the full published 18-35% range.")

## 4. Headline metrics: precision, recall, F1, PR-AUC, ROC-AUC

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

test_m = classification_metrics(y_test, p_test)
val_m = classification_metrics(y_val, p_val)
flag_test = p_test >= THRESHOLD

headline = pd.DataFrame({
    "validation": {
        "PR-AUC": val_m["pr_auc"], "ROC-AUC": val_m["roc_auc"],
        "Brier": val_m["brier"], "log-loss": val_m["log_loss"],
        "base rate": val_m["base_rate"], "n": len(y_val),
    },
    "TEST": {
        "PR-AUC": test_m["pr_auc"], "ROC-AUC": test_m["roc_auc"],
        "Brier": test_m["brier"], "log-loss": test_m["log_loss"],
        "base rate": test_m["base_rate"], "n": len(y_test),
    },
}).T
display(headline.round(4))

print(f"\nAt the frozen threshold of {THRESHOLD:.3f}:")
print(f"  precision  {precision_score(y_test, flag_test):.4f}")
print(f"  recall     {recall_score(y_test, flag_test):.4f}")
print(f"  F1         {f1_score(y_test, flag_test):.4f}")
print(f"  flag rate  {flag_test.mean():.4f}  ({flag_test.sum():,} of {len(y_test):,} orders)")
print(f"\nNo-skill PR-AUC on this split is the base rate itself: {y_test.mean():.4f}")
print(f"PR-AUC lift over no-skill: {test_m['pr_auc'] / y_test.mean():.2f}x")

In [ ]:
drift = pd.DataFrame([
    {"metric": "PR-AUC", "val": val_m["pr_auc"], "test": test_m["pr_auc"]},
    {"metric": "ROC-AUC", "val": val_m["roc_auc"], "test": test_m["roc_auc"]},
    {"metric": "Brier", "val": val_m["brier"], "test": test_m["brier"]},
])
drift["delta"] = drift.test - drift["val"]
drift["pct_change"] = (drift.delta / drift["val"] * 100).round(1)
display(drift.round(4).style.hide(axis="index"))

print("""Validation-to-test drift is the single most informative number on this page.
A large drop would mean the model was selected on validation noise. A large RISE
would be equally suspicious. Both splits are out-of-time relative to training, so
the comparison is like for like -- except that the base rate differs, which is
exactly what section 10 is about.""")

## 5. Confusion matrix at cost-optimal threshold

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, flag_test)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", cbar=False, ax=ax[0],
            xticklabels=["allow COD", "flag"], yticklabels=["delivered", "RTO"])
ax[0].set_title(f"Confusion matrix at the cost-optimal threshold ({THRESHOLD:.3f})")
ax[0].set_ylabel("actual")
ax[0].set_xlabel("decision")

cm_half = confusion_matrix(y_test, p_test >= 0.5)
sns.heatmap(cm_half, annot=True, fmt=",d", cmap="Oranges", cbar=False, ax=ax[1],
            xticklabels=["allow COD", "flag"], yticklabels=["delivered", "RTO"])
ax[1].set_title("What the default 0.5 threshold would have done")
ax[1].set_ylabel("actual")
ax[1].set_xlabel("decision")
plt.tight_layout()
plt.savefig(FIG / "05_confusion_matrix.png", bbox_inches="tight")
plt.show()

print(f"""At {THRESHOLD:.3f}:  TP {tp:,}   FP {fp:,}   FN {fn:,}   TN {tn:,}

  {tp:,} returns caught, {fn:,} missed.
  {fp:,} good orders discouraged -- the cost the flag rate buys.

At 0.500:  TP {cm_half[1,1]:,}   FP {cm_half[0,1]:,}   FN {cm_half[1,0]:,}   TN {cm_half[0,0]:,}

The 0.5 threshold is not a neutral default. It is the right threshold only when a
false positive and a false negative cost the same amount, which here they
emphatically do not. Section 7 prices the difference.""")

## 6. Calibration: Brier score + reliability diagram

In [ ]:
curve_test, ece_test = reliability_curve(y_test, p_test, n_bins=10)
curve_val, ece_val = reliability_curve(y_val, p_val, n_bins=10)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))

panels = [
    (curve_test, ece_test, test_m["brier"], "TEST", "#c44"),
    (curve_val, ece_val, val_m["brier"], "validation", "#68a"),
]
for a, (curve, ece, brier, name, colour) in zip(ax[:2], panels):
    hi = curve.mean_predicted.max() * 1.05
    a.plot([0, hi], [0, hi], "k--", lw=1, label="perfect calibration")
    a.errorbar(curve.mean_predicted, curve.observed_rate,
               yerr=[curve.observed_rate - curve.ci_lo, curve.ci_hi - curve.observed_rate],
               fmt="o-", color=colour, capsize=3, lw=1.8, label=name)
    a.axvline(THRESHOLD, ls=":", c="#4a8", lw=1.6, label=f"operating point {THRESHOLD:.2f}")
    a.set_xlabel("mean predicted probability")
    a.set_ylabel("observed RTO rate")
    a.set_title(f"Reliability diagram - {name}\n"
                f"Brier {brier:.4f}   ECE {ece:.4f}")
    a.legend(fontsize=8)

ax[2].hist(p_test, bins=50, color="#68a")
ax[2].axvline(THRESHOLD, ls=":", c="#4a8", lw=2, label=f"threshold {THRESHOLD:.2f}")
ax[2].axvline(LOW_CUT, ls="--", c="#999", lw=1.4, label=f"tier cuts")
ax[2].axvline(HIGH_CUT, ls="--", c="#999", lw=1.4)
ax[2].set_yscale("log")
ax[2].set_xlabel("predicted P(RTO)")
ax[2].set_title("Score distribution on test (log count)")
ax[2].legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG / "05_calibration.png", bbox_inches="tight")
plt.show()

display(curve_test.round(4).style.hide(axis="index"))

In [ ]:
worst = curve_test.loc[curve_test.gap.abs().idxmax()]
near_op = curve_test.iloc[(curve_test.mean_predicted - THRESHOLD).abs().argsort()[:1]].iloc[0]

print(f"""CALIBRATION READ

  Brier (test)                {test_m['brier']:.4f}
  Expected Calibration Error  {ece_test:.4f}
  worst bin                   predicted {worst.mean_predicted:.3f} vs observed {worst.observed_rate:.3f}
                              (gap {worst.gap:+.3f} on n={int(worst.n):,})
  bin nearest the threshold   predicted {near_op.mean_predicted:.3f} vs observed {near_op.observed_rate:.3f}
                              (gap {near_op.gap:+.3f} on n={int(near_op.n):,})

Most submissions stop at Brier, or skip calibration entirely. It matters here
because the threshold is a COST minimum: if the model says 0.30 and the truth is
0.45, the cost curve is minimised at the wrong place and the merchant pays for it.

The bin nearest the operating point is the one that decides money. Its gap is
reported above whether it flatters us or not.""")

## 7. Cost model in rupees (FN = shipping burned, FP = lost margin)

In [ ]:
test_at_thr = binary_cost(y_test, p_test, THRESHOLD, test_values, COSTS)
test_at_half = binary_cost(y_test, p_test, 0.50, test_values, COSTS)
baselines = no_model_baselines(y_test, test_values, COSTS)

n = len(y_test)
comparison = pd.DataFrame([
    {"policy": "allow COD on everything (no model)",
     "total_cost_inr": baselines["allow_everything"],
     "cost_per_order_inr": baselines["allow_everything"] / n},
    {"policy": "disable COD on everything (no model)",
     "total_cost_inr": baselines["block_everything"],
     "cost_per_order_inr": baselines["block_everything"] / n},
    {"policy": "model @ default 0.5 threshold",
     "total_cost_inr": test_at_half["total_cost_inr"],
     "cost_per_order_inr": test_at_half["cost_per_order_inr"]},
    {"policy": f"model @ cost-optimal {THRESHOLD:.3f} (SHIPPED)",
     "total_cost_inr": test_at_thr["total_cost_inr"],
     "cost_per_order_inr": test_at_thr["cost_per_order_inr"]},
])
comparison["vs_best_no_model"] = (
    comparison.total_cost_inr - min(baselines.values()))
display(comparison.round(2).style.hide(axis="index"))

saving_vs_nomodel = min(baselines.values()) - test_at_thr["total_cost_inr"]
saving_vs_half = test_at_half["total_cost_inr"] - test_at_thr["total_cost_inr"]

print(f"""
On {n:,} held-out test orders:

  best no-model policy        Rs{min(baselines.values()):>12,.0f}
  model at the 0.5 default    Rs{test_at_half['total_cost_inr']:>12,.0f}
  model at the cost minimum   Rs{test_at_thr['total_cost_inr']:>12,.0f}

  saving vs the best no-model policy   Rs{saving_vs_nomodel:>10,.0f}  ({saving_vs_nomodel / n:.2f}/order)
  saving vs using 0.5                  Rs{saving_vs_half:>10,.0f}  ({saving_vs_half / n:.2f}/order)

THE COST OF THE DEFAULT THRESHOLD: choosing 0.5 out of habit rather than the cost
minimum would have cost an extra Rs{saving_vs_half:,.0f} on these {n:,} orders alone.""")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))

sweep_test = threshold_sweep(y_test, p_test, test_values, COSTS)
ax[0].plot(sweep_test.threshold, sweep_test.cost_per_order_inr, lw=2, label="total")
ax[0].plot(sweep_test.threshold, sweep_test.cost_false_negatives_inr / n,
           lw=1.3, ls="--", label="false negatives (shipping burned)")
ax[0].plot(sweep_test.threshold, sweep_test.cost_false_positives_inr / n,
           lw=1.3, ls="--", label="false positives (lost margin)")
ax[0].axvline(THRESHOLD, c="#4a8", lw=2,
              label=f"frozen threshold {THRESHOLD:.3f} (chosen on val)")
ax[0].axvline(0.5, c="#c44", ls=":", lw=2, label="default 0.5")
best_test_thr = sweep_test.loc[sweep_test.total_cost_inr.idxmin(), "threshold"]
ax[0].axvline(best_test_thr, c="k", ls="-.", lw=1.2,
              label=f"test-optimal {best_test_thr:.3f} (not used)")
ax[0].set_xlabel("threshold")
ax[0].set_ylabel("cost per order (INR)")
ax[0].set_title("Rupee cost vs threshold, on TEST")
ax[0].legend(fontsize=7.5)

ax[1].plot(sweep_test.recall, sweep_test.precision, lw=2)
op = sweep_test.loc[(sweep_test.threshold - THRESHOLD).abs().idxmin()]
ax[1].scatter([op.recall], [op.precision], s=90, c="#4a8", zorder=4,
              label=f"operating point ({op.recall:.2f}, {op.precision:.2f})")
oph = sweep_test.loc[(sweep_test.threshold - 0.5).abs().idxmin()]
ax[1].scatter([oph.recall], [oph.precision], s=90, c="#c44", zorder=4,
              label=f"0.5 default ({oph.recall:.2f}, {oph.precision:.2f})")
ax[1].axhline(y_test.mean(), ls="--", c="k", lw=1, label="no-skill precision")
ax[1].set_xlabel("recall")
ax[1].set_ylabel("precision")
ax[1].set_title("Where the cost minimum sits on the PR curve")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG / "05_cost_curve.png", bbox_inches="tight")
plt.show()

print(f"""The frozen threshold ({THRESHOLD:.3f}) and the threshold that would have been
optimal on test ({best_test_thr:.3f}) differ by {abs(THRESHOLD - best_test_thr):.3f}. The test-optimal
value is plotted for honesty and NOT used -- picking it would be choosing an
operating point on the test set, which is the thing this whole protocol exists to
avoid. The cost difference between them is
Rs{sweep_test.total_cost_inr.min() - test_at_thr['total_cost_inr']:,.0f} across {n:,} orders, which is the price of doing this properly.""")

## 8. Threshold sweep vs cost - choose the three tiers

In [ ]:
# fn_cost_inr is marked `assumed` with a stated 150-250 range, and margin_rate
# with 0.15-0.35. The honest report is not one threshold but how far the answer
# moves when the assumptions do.
sens = sensitivity(y_test, p_test, test_values, COSTS)

pivot_thr = sens.pivot(index="fn_cost_inr", columns="margin_rate",
                       values="optimal_threshold")
pivot_cost = sens.pivot(index="fn_cost_inr", columns="margin_rate",
                        values="cost_per_order_inr")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
sns.heatmap(pivot_thr, annot=True, fmt=".3f", cmap="RdYlGn_r", ax=ax[0],
            cbar_kws={"label": "cost-optimal threshold"})
ax[0].set_title("Optimal threshold across the assumed cost ranges")
sns.heatmap(pivot_cost, annot=True, fmt=".1f", cmap="Blues", ax=ax[1],
            cbar_kws={"label": "cost per order (INR)"})
ax[1].set_title("Cost per order at that threshold")
plt.tight_layout()
plt.savefig(FIG / "05_cost_sensitivity.png", bbox_inches="tight")
plt.show()

print(f"""SENSITIVITY

  threshold range across the assumed grid : {sens.optimal_threshold.min():.3f} - {sens.optimal_threshold.max():.3f}
  our frozen threshold                    : {THRESHOLD:.3f}
  recall range                            : {sens.recall.min():.3f} - {sens.recall.max():.3f}

The threshold is genuinely sensitive to the FN/FP cost ratio, and pretending
otherwise would be the dishonest move. What this table gives a merchant is the
ability to substitute THEIR shipping cost and THEIR margin and read off the
threshold, rather than inheriting ours.""")

In [ ]:
# The three-tier cut points, chosen on validation, shown on the validation cost
# surface they were chosen from.
fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))

grid_lo = np.sort(tier_surface.low_cut.unique())
grid_hi = np.sort(tier_surface.high_cut.unique())
Z = (tier_surface.pivot(index="low_cut", columns="high_cut",
                        values="cost_per_order_inr"))
im = ax[0].pcolormesh(Z.columns, Z.index, Z.values, cmap="viridis_r", shading="auto")
ax[0].scatter([HIGH_CUT], [LOW_CUT], marker="*", s=280, c="#fc0",
              edgecolor="k", zorder=5, label="chosen cuts")
ax[0].set_xlabel("high cut (disable COD above this)")
ax[0].set_ylabel("low cut (allow COD below this)")
ax[0].set_title("Validation cost surface over the two cut points")
ax[0].legend(fontsize=8)
plt.colorbar(im, ax=ax[0], label="cost per order (INR)")

two_tier = binary_cost(y_test, p_test, THRESHOLD, test_values, COSTS)
three_tier = three_tier_cost(y_test, p_test, test_values, LOW_CUT, HIGH_CUT, COSTS)
bars = pd.Series({
    "allow all": baselines["allow_everything"] / n,
    "block all": baselines["block_everything"] / n,
    "2-tier @ 0.5": test_at_half["cost_per_order_inr"],
    "2-tier @ cost min": two_tier["cost_per_order_inr"],
    "3-tier policy": three_tier["cost_per_order_inr"],
})
bars.plot(kind="bar", ax=ax[1], rot=20,
          color=["#999", "#999", "#c44", "#68a", "#4a8"])
ax[1].set_ylabel("cost per order (INR)")
ax[1].set_title("Policy comparison on TEST")
for i, v in enumerate(bars):
    ax[1].text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(FIG / "05_tier_policy.png", bbox_inches="tight")
plt.show()

## 9. Three-tier decision policy: allow / COD fee / disable COD

In [ ]:
tiers_test = apply_tiers(p_test, LOW_CUT, HIGH_CUT)
tier_tbl = pd.DataFrame({
    "tier": TIERS,
    "action": ["allow COD", "charge a COD fee", "disable COD, offer prepaid"],
    "score_range": [f"< {LOW_CUT:.3f}", f"{LOW_CUT:.3f} - {HIGH_CUT:.3f}",
                    f">= {HIGH_CUT:.3f}"],
})
tier_tbl["n_orders"] = [int((tiers_test == t).sum()) for t in TIERS]
tier_tbl["share"] = tier_tbl.n_orders / len(y_test)
tier_tbl["actual_rto_rate"] = [
    float(y_test[tiers_test == t].mean()) if (tiers_test == t).any() else np.nan
    for t in TIERS]
tier_tbl["mean_order_value"] = [
    float(test_values[tiers_test == t].mean()) if (tiers_test == t).any() else np.nan
    for t in TIERS]
tier_tbl["lift_vs_base"] = tier_tbl.actual_rto_rate / y_test.mean()
display(tier_tbl.round(4).style.hide(axis="index"))

print(f"""
The policy separates the population, which is the entire point:

  allow tier    {tier_tbl.actual_rto_rate.iat[0]:.1%} actual RTO rate  ({tier_tbl.share.iat[0]:.0%} of orders, no friction)
  fee tier      {tier_tbl.actual_rto_rate.iat[1]:.1%}                  ({tier_tbl.share.iat[1]:.0%} of orders, risk priced)
  disable tier  {tier_tbl.actual_rto_rate.iat[2]:.1%}                  ({tier_tbl.share.iat[2]:.0%} of orders, loss avoided)

A binary blocker throws away the middle. The fee tier keeps those sales at a price
that covers their risk instead of refusing them -- which is why the three-tier
policy costs less than the two-tier one even though it flags more orders.""")

In [ ]:
print("""DEFENCE-ONLY BOUNDARY

  This notebook computes what each decision WOULD cost. It does not take any.

  The system has no code path that captures a payment, issues a refund, blocks an
  account, cancels an order or contacts a customer. `apply_tiers` returns three
  strings. Acting on them is the merchant's decision, made in the merchant's own
  systems.

  Nothing here is offense-capable: the model consumes order attributes and emits
  a probability, a tier and a list of reasons. There is no capability to
  repurpose.""")

## 10. Prevalence-shift study (18% - 35%, the real Indian city range)

In [ ]:
print("""THE CENTREPIECE.

Published Indian city RTO rates run from 18% (Vadodara) to 35% (Patna). This model
was trained where 17% of orders return and tested where 14.7% do. Deploying it in
Patna means asking it to operate at more than twice the base rate it learned.

Two failure modes hide behind a single aggregate number:

  1. RANKING degrades -- the model can no longer tell risky from safe.
  2. CALIBRATION degrades while ranking survives -- the model still ranks fine,
     but its probabilities are wrong, so the COST-OPTIMAL THRESHOLD is in the
     wrong place and the merchant pays anyway.

The second is the dangerous one, because PR-AUC alone will not show it. We measure
both, and we test the standard remedy for it.

Method: the test split is RESAMPLED (subsampling only -- never duplicated, never
synthesised) to each target prevalence, so every scored row is a real row with a
real prediction. n shrinks at the extremes, so every point carries a bootstrap
band.""")

RATES = np.round(np.arange(0.14, 0.401, 0.01), 3)
TRAIN_RATE = 0.1696

curve = prevalence_curve(y_test, p_test, RATES, seed=SEED, n_boot=200)
display(curve[curve.target_prevalence.isin([0.18, 0.22, 0.26, 0.30, 0.35, 0.40])][
    ["target_prevalence", "n", "pr_auc", "pr_auc_lift", "roc_auc", "brier"]
].round(4).style.hide(axis="index"))

In [ ]:
# Cost under shift: the frozen threshold, an oracle re-optimised threshold, the
# prior-corrected threshold, and the no-model baselines.
rows = []
for r in RATES:
    idx = resample_to_prevalence(y_test, r, seed=SEED)
    ys, ps, vs = y_test[idx], p_test[idx], test_values[idx]

    frozen = binary_cost(ys, ps, THRESHOLD, vs, COSTS)
    oracle_thr, sw = optimal_threshold(ys, ps, vs, COSTS)
    oracle = sw.loc[sw.threshold == oracle_thr].iloc[0]

    # The remedy: rescale probabilities for the new base rate, then re-derive the
    # threshold from the CORRECTED scores. Needs only the new prevalence, which a
    # merchant knows from their own returns.
    ps_corr = prior_shift_correction(ps, TRAIN_RATE, r)
    corr_thr, sw_c = optimal_threshold(ys, ps_corr, vs, COSTS)
    corrected = sw_c.loc[sw_c.threshold == corr_thr].iloc[0]

    base = no_model_baselines(ys, vs, COSTS)
    _, ece_r = reliability_curve(ys, ps, n_bins=10)
    _, ece_c = reliability_curve(ys, ps_corr, n_bins=10)

    rows.append({
        "prevalence": r, "n": len(ys),
        "cost_frozen": frozen["cost_per_order_inr"],
        "cost_oracle": float(oracle.cost_per_order_inr),
        "cost_prior_corrected": float(corrected.cost_per_order_inr),
        "regret_frozen": frozen["cost_per_order_inr"] - float(oracle.cost_per_order_inr),
        "regret_corrected": float(corrected.cost_per_order_inr) - float(oracle.cost_per_order_inr),
        "cost_allow_all": base["allow_everything"] / len(ys),
        "cost_block_all": base["block_everything"] / len(ys),
        "oracle_threshold": oracle_thr,
        "recall_frozen": frozen["recall"], "precision_frozen": frozen["precision"],
        "ece_raw": ece_r, "ece_corrected": ece_c,
    })
shift = pd.DataFrame(rows)
shift["best_no_model"] = shift[["cost_allow_all", "cost_block_all"]].min(axis=1)
shift["beats_no_model"] = shift.cost_frozen < shift.best_no_model
display(shift[shift.prevalence.isin([0.18, 0.22, 0.26, 0.30, 0.35, 0.40])].round(3)
        .style.hide(axis="index"))

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(15, 9))

a = ax[0, 0]
a.plot(curve.actual_prevalence, curve.pr_auc, "o-", lw=2, label="PR-AUC")
a.fill_between(curve.actual_prevalence, curve.pr_auc_lo, curve.pr_auc_hi, alpha=0.2)
a.plot(curve.actual_prevalence, curve.actual_prevalence, "k--", lw=1.2,
       label="no-skill floor (= base rate)")
a.axvspan(0.18, 0.35, alpha=0.08, color="green")
a.axvline(y_test.mean(), ls=":", c="#c44", label="test base rate")
a.set_xlabel("base rate")
a.set_ylabel("PR-AUC")
a.set_title("PR-AUC rises with prevalence - which is why it must not be read alone\n"
            "(green = published 18-35% Indian city range)")
a.legend(fontsize=8)

a = ax[0, 1]
a.plot(curve.actual_prevalence, curve.pr_auc_lift, "o-", lw=2, c="#4a8")
a.axhline(1.0, ls="--", c="k", lw=1.2, label="no skill")
a.axvspan(0.18, 0.35, alpha=0.08, color="green")
a.set_xlabel("base rate")
a.set_ylabel("PR-AUC / base rate")
a.set_title("Lift over the no-skill floor - the comparable quantity")
a.legend(fontsize=8)

a = ax[1, 0]
a.plot(shift.prevalence, shift.cost_frozen, "o-", lw=2, c="#c44",
       label=f"frozen threshold {THRESHOLD:.2f}")
a.plot(shift.prevalence, shift.cost_prior_corrected, "s-", lw=2, c="#4a8",
       label="prior-corrected threshold")
a.plot(shift.prevalence, shift.cost_oracle, "--", lw=1.5, c="k",
       label="oracle threshold (unattainable)")
a.plot(shift.prevalence, shift.best_no_model, ":", lw=2, c="#999",
       label="best no-model policy")
a.axvspan(0.18, 0.35, alpha=0.08, color="green")
a.set_xlabel("base rate")
a.set_ylabel("cost per order (INR)")
a.set_title("THE DEGRADATION CURVE - rupee cost under prevalence shift")
a.legend(fontsize=8)

a = ax[1, 1]
a.plot(shift.prevalence, shift.ece_raw, "o-", lw=2, c="#c44", label="raw scores")
a.plot(shift.prevalence, shift.ece_corrected, "s-", lw=2, c="#4a8",
       label="prior-corrected")
a.axvspan(0.18, 0.35, alpha=0.08, color="green")
a.set_xlabel("base rate")
a.set_ylabel("Expected Calibration Error")
a.set_title("Calibration is what breaks first - and the correction fixes it")
a.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG / "05_prevalence_shift.png", bbox_inches="tight")
plt.show()

In [ ]:
in_range = shift[(shift.prevalence >= 0.18) & (shift.prevalence <= 0.35)]
worst = in_range.loc[in_range.regret_frozen.idxmax()]
never_worse = bool(in_range.beats_no_model.all())
lift_min = curve[(curve.actual_prevalence >= 0.18) &
                 (curve.actual_prevalence <= 0.35)].pr_auc_lift.min()

print(f"""PREVALENCE-SHIFT VERDICT, across the published 18-35% range

  ranking            PR-AUC lift never falls below {lift_min:.2f}x the no-skill floor
  beats no-model     at every prevalence in range: {never_worse}
  worst regret       Rs{worst.regret_frozen:.2f}/order at a {worst.prevalence:.0%} base rate
                     (frozen Rs{worst.cost_frozen:.2f} vs oracle Rs{worst.cost_oracle:.2f})
  with correction    worst regret falls to Rs{in_range.regret_corrected.max():.2f}/order
  calibration        ECE rises from {shift.ece_raw.iloc[0]:.3f} to {in_range.ece_raw.max():.3f} untreated,
                     and stays at {in_range.ece_corrected.max():.3f} with the prior correction

WHAT THIS ANSWERS

  A model can hold its ranking and still lose money under prevalence shift,
  because the threshold stops being the cost minimum. That is the failure mode
  that a PR-AUC-only report cannot see.

  Here the RANKING is stable across the whole published range -- the model does
  not collapse. What degrades is CALIBRATION, and therefore the operating point.
  The regret is bounded and measured above rather than assumed away.

  The remedy is one line of arithmetic (src.evaluate.prior_shift_correction): a
  merchant who knows their own RTO rate can rescale the scores for their own base
  rate before thresholding. That requires no retraining and no new labels.

  This is reported at every prevalence, including the ones where we do worst.""")

## 11. SHAP explanations & risk reasons

In [ ]:
import shap

pre = FINAL_PIPE.named_steps["pre"]
clf = FINAL_PIPE.named_steps["clf"]
X_test = to_design_matrix(pre, test)
feat_names = list(X_test.columns)

# Subsample for tractability; SHAP on 10,000 x 66 with a forest is slow and adds
# nothing over a well-sized sample.
rng = np.random.default_rng(SEED)
sub = np.sort(rng.choice(len(X_test), size=min(2000, len(X_test)), replace=False))
X_sub = X_test.iloc[sub]

try:
    explainer = shap.TreeExplainer(clf)
    sv = explainer.shap_values(X_sub)
    if isinstance(sv, list):
        sv = sv[1]
    elif sv.ndim == 3:
        sv = sv[:, :, 1]
except Exception:
    background = shap.sample(X_test, 100, random_state=SEED)
    explainer = shap.Explainer(lambda d: clf.predict_proba(d)[:, 1], background)
    sv = explainer(X_sub).values

sv = np.asarray(sv, dtype=float)
print(f"SHAP values: {sv.shape} on {len(X_sub):,} sampled test orders")

In [ ]:
mean_abs = pd.Series(np.abs(sv).mean(axis=0), index=feat_names).sort_values()

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
mean_abs.tail(18).plot(kind="barh", ax=ax[0], color="#68a")
ax[0].set_xlabel("mean |SHAP value|")
ax[0].set_title("Global feature importance on TEST")

plt.sca(ax[1])
shap.summary_plot(sv, X_sub, feature_names=feat_names, max_display=15,
                  show=False, plot_size=None)
ax[1].set_title("Direction and spread of each feature's effect")
plt.tight_layout()
plt.savefig(FIG / "05_shap_summary.png", bbox_inches="tight")
plt.show()

print("\nTop 12 drivers by mean |SHAP|:")
display(mean_abs.sort_values(ascending=False).head(12).round(5).to_frame("mean_abs_shap"))

In [ ]:
# The features 02 declared weak BEFORE any model saw them. This is the check on
# that prediction, and it goes in the exception list either way.
declared_weak = ["order_velocity_24h", "account_age_days", "addr_gibberish_score"]
weak_check = pd.DataFrame({
    "feature": declared_weak,
    "mean_abs_shap": [float(mean_abs.get(f, np.nan)) for f in declared_weak],
})
weak_check["rank_of_66"] = [
    int((mean_abs > mean_abs.get(f, 0)).sum()) + 1 if f in mean_abs.index else np.nan
    for f in declared_weak]
weak_check["share_of_top_feature"] = weak_check.mean_abs_shap / mean_abs.max()
display(weak_check.round(5).style.hide(axis="index"))

print("""02 predicted these three would contribute little, on univariate AUC alone,
before any model was fitted. SHAP on the test set is the verdict on that
prediction. Whatever it says goes in section 13 -- a confirmed weak feature is
reported as a weak feature, not quietly dropped from the write-up.""")

In [ ]:
# Human-readable risk reasons: what a merchant would actually be shown.
order_idx = np.argsort(-p_test[sub])[:4].tolist() + np.argsort(p_test[sub])[:2].tolist()

print("=" * 78)
print("EXAMPLE SCORED ORDERS -- what the merchant sees")
print("=" * 78)
for i in order_idx:
    g = test.iloc[sub[i]]
    score = p_test[sub[i]]
    tier = apply_tiers([score], LOW_CUT, HIGH_CUT)[0]
    action = {ALLOW: "allow COD", FEE: "charge a COD fee",
              BLOCK: "disable COD, offer prepaid"}[tier]
    reasons = risk_reasons(sv[i], feat_names, top_k=3)
    print(f"""
  order {g.order_id}   {g.city}, {g.state} ({g.pincode_tier})
    Rs{g.order_value:,.0f} {g.category}, {g.payment_mode}, {g.discount_pct:.0%} off
    RISK SCORE   {score:.3f}     RECOMMENDATION   {action}
    because:     {'; '.join(reasons) if reasons else 'no upward risk factors'}
    actual outcome (hidden from the model): {'RETURNED' if g.rto else 'delivered'}""")

print("""

The reasons come from SHAP, not from a hand-written rule list, so they track what
the model actually used. They are phrased as facts about the order rather than as
statements about the customer -- a merchant can act on "no house number in the
address"; nobody can act on "feature 17 = 0.83".""")

## 12. Model comparison plots (all 10 + tuned)

In [ ]:
# The 11 benchmark entries and the tuned finalists are compared on VALIDATION,
# where model selection legitimately happened. Scoring all of them on test and
# then discussing the ranking would be test-set shopping after the fact.
# The selected model's TEST point is overlaid, so val -> test drift is visible
# for the one model that earned a test score.
comp_rows = []
for k, p in ALL_VAL_PRED.items():
    m = classification_metrics(y_val, p)
    if k in tune["finalists"] or "__" in k:
        label = (f"{zoo[k.split('__')[0]].label} ({k.split('__')[1]})"
                 if "__" in k else zoo[k].label)
        group = "tuned" if "__" in k else "untuned"
    else:
        label, group = zoo[k].label, "untuned"
    comp_rows.append({"key": k, "label": label, "group": group,
                      "pr_auc": m["pr_auc"], "roc_auc": m["roc_auc"],
                      "brier": m["brier"]})
comp = pd.DataFrame(comp_rows).sort_values("pr_auc")

fig, ax = plt.subplots(1, 2, figsize=(15, 6.5))
colors = {"untuned": "#999", "tuned": "#4a8"}
ax[0].barh(comp.label, comp.pr_auc, color=[colors[g] for g in comp.group])
ax[0].axvline(bench["bayes_ceiling_val"]["pr_auc"], ls="--", c="#c44", lw=1.5,
              label="Bayes ceiling (val)")
ax[0].axvline(test_m["pr_auc"], ls="-.", c="#06c", lw=1.8,
              label=f"SHIPPED model on TEST ({test_m['pr_auc']:.3f})")
ax[0].set_xlabel("validation PR-AUC")
ax[0].set_title("All benchmark entries + tuned finalists")
ax[0].legend(fontsize=8, loc="lower right")

sc = ax[1].scatter(comp.brier, comp.pr_auc,
                   c=[colors[g] for g in comp.group], s=80)
for _, r in comp.iterrows():
    ax[1].annotate(r.label, (r.brier, r.pr_auc), fontsize=6.5,
                   xytext=(4, 3), textcoords="offset points")
ax[1].scatter([test_m["brier"]], [test_m["pr_auc"]], marker="*", s=340, c="#06c",
              edgecolor="k", zorder=5, label="SHIPPED model, TEST")
ax[1].set_xlabel("Brier score (lower is better)")
ax[1].set_ylabel("PR-AUC")
ax[1].set_title("Ranking quality vs calibration - the shipped model needs both")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG / "05_model_comparison.png", bbox_inches="tight")
plt.show()

display(comp.sort_values("pr_auc", ascending=False).round(4)
        .style.hide(axis="index"))

## 13. Honest exception list - where it underperforms

In [ ]:
train_pincodes = set(pd.read_parquet(
    ROOT / "data/processed/train.parquet", columns=["pincode"]).pincode.unique())

segments = {
    "ALL TEST ORDERS": np.ones(len(test), dtype=bool),
    "COD orders": test.is_cod.to_numpy(),
    "prepaid orders": ~test.is_cod.to_numpy(),
    "first-time customers": (test.has_history == 0).to_numpy(),
    "customers with history": (test.has_history == 1).to_numpy(),
    "pincode UNSEEN in train": ~test.pincode.isin(train_pincodes).to_numpy(),
    "pincode seen in train": test.pincode.isin(train_pincodes).to_numpy(),
    "metro": (test.pincode_tier == "metro").to_numpy(),
    "tier_1": (test.pincode_tier == "tier_1").to_numpy(),
    "tier_2": (test.pincode_tier == "tier_2").to_numpy(),
    "tier_3": (test.pincode_tier == "tier_3").to_numpy(),
    "order < Rs500": (test.order_value < 500).to_numpy(),
    "order Rs500-1000": ((test.order_value >= 500) & (test.order_value < 1000)).to_numpy(),
    "order > Rs1000": (test.order_value >= 1000).to_numpy(),
    "top-decile order value": (test.order_value >= test.order_value.quantile(0.9)).to_numpy(),
    "fashion": (test.category == "fashion").to_numpy(),
    "electronics": (test.category == "electronics").to_numpy(),
    "festive window": test.is_festive.to_numpy(),
    "alternate address": test.is_alternate_address.to_numpy(),
    "long ETA (>= 6 days)": (test.delivery_days_est >= 6).to_numpy(),
}

seg = segment_report(test, y_test, p_test, THRESHOLD, segments, min_n=100)
overall_lift = seg.loc[seg.segment == "ALL TEST ORDERS", "pr_auc_lift"].iat[0]
seg["lift_vs_overall"] = seg.pr_auc_lift / overall_lift
display(seg.round(4).style.background_gradient(
    subset=["pr_auc_lift"], cmap="RdYlGn").hide(axis="index"))

In [ ]:
scored = seg.dropna(subset=["pr_auc_lift"]).copy()
scored = scored[scored.segment != "ALL TEST ORDERS"]
weak_segments = scored.nsmallest(5, "pr_auc_lift")
miscal = scored.reindex(scored.mean_pred_vs_actual.abs().sort_values(
    ascending=False).index).head(5)

fig, ax = plt.subplots(1, 2, figsize=(15, 5.2))
s = scored.sort_values("pr_auc_lift")
ax[0].barh(s.segment, s.pr_auc_lift,
           color=["#c44" if v < 1.0 else "#68a" for v in s.pr_auc_lift])
ax[0].axvline(1.0, ls="--", c="k", lw=1.2, label="no skill")
ax[0].axvline(overall_lift, ls=":", c="#4a8", lw=1.6, label="overall")
ax[0].set_xlabel("PR-AUC lift over that segment's own base rate")
ax[0].set_title("Where the model is strong and where it is not")
ax[0].legend(fontsize=8)

s2 = scored.sort_values("mean_pred_vs_actual")
ax[1].barh(s2.segment, s2.mean_pred_vs_actual,
           color=["#c44" if v > 0 else "#68a" for v in s2.mean_pred_vs_actual])
ax[1].axvline(0, c="k", lw=1.2)
ax[1].set_xlabel("mean predicted - actual RTO rate")
ax[1].set_title("Calibration bias by segment (right = overcharges good customers)")
plt.tight_layout()
plt.savefig(FIG / "05_exception_list.png", bbox_inches="tight")
plt.show()

In [ ]:
print("=" * 78)
print("HONEST EXCEPTION LIST")
print("=" * 78)
print("""
Named and quantified. Every one of these is a place a merchant should not trust
this model without a second look.
""")

print("1. WEAKEST SEGMENTS BY RANKING (PR-AUC lift over the segment's own base rate)\n")
for _, r in weak_segments.iterrows():
    print(f"   {r.segment:<28} lift {r.pr_auc_lift:.2f}x   n={int(r.n):,}  "
          f"base {r.base_rate:.3f}  recall {r.recall:.3f}")

print("\n2. WORST CALIBRATION BIAS (mean predicted minus actual)\n")
for _, r in miscal.iterrows():
    direction = "OVER-estimates risk" if r.mean_pred_vs_actual > 0 else "UNDER-estimates risk"
    print(f"   {r.segment:<28} {r.mean_pred_vs_actual:+.4f}  {direction}   n={int(r.n):,}")

too_small = seg[seg.get("note").notna()] if "note" in seg.columns else pd.DataFrame()
if len(too_small):
    print("\n3. SEGMENTS TOO SMALL TO JUDGE (reported rather than omitted)\n")
    for _, r in too_small.iterrows():
        print(f"   {r.segment:<28} n={int(r.n):,} - {r.note}")

print(f"""
4. STRUCTURAL LIMITATIONS

   Prepaid orders. The model is a COD gate. Prepaid RTO runs near 1.4%, so there
   is almost no signal to find and almost nothing to save. It should not be used
   to make prepaid decisions.

   First-time customers ({(test.has_history == 0).mean():.0%} of test orders). No purchase history
   exists, so the model works from geography, address quality and basket alone.
   02 measured the cost of that dilution at 0.029 AUC in advance.

   Unseen pincodes. A pincode absent from training falls back to the smoothed
   prior. Performance on that slice is in the table above; it is a real operating
   condition for a merchant expanding into new corridors.

   Weak features, as predicted in 02 before any model was fitted:
     order_velocity_24h  - 99.0% of orders have velocity 0
     account_age_days    - near-collinear with has_history
     addr_gibberish_score- fires on genuine low-vowel Indian place names
   Their SHAP contributions are in section 11. They stayed in because a tree can
   use a thin tail conditionally, but no claim is made for them.

5. THE LARGEST LIMITATION OF ALL

   This is synthetic data. It is built on the real India Post pincode directory
   and calibrated against published Indian RTO statistics by assertion, but the
   conditional structure is our modelling choice. Performance here is evidence of
   method, not evidence of production performance. No claim of validation on real
   merchant data is made anywhere in this submission.""")

## 14. Persist final model + metrics

In [ ]:
import joblib

final_path = MODELS / "final_model.joblib"
joblib.dump({
    "pipeline": FINAL_PIPE,
    "model_key": FINAL_KEY,
    "threshold": THRESHOLD,
    "low_cut": LOW_CUT,
    "high_cut": HIGH_CUT,
    "train_base_rate": TRAIN_RATE,
    "feature_columns": COLS,
    "cost_params": COSTS.__dict__,
    "seed": SEED,
}, final_path)

test_pred = pd.DataFrame({
    "order_id": test.order_id.to_numpy(),
    "y_true": y_test,
    "p_rto": p_test,
    "tier": tiers_test,
})
test_pred.to_parquet(RES / "05_test_predictions.parquet", index=False)
shift.to_csv(RES / "05_prevalence_shift.csv", index=False)
seg.to_csv(RES / "05_exception_list.csv", index=False)
sens.to_csv(RES / "05_cost_sensitivity.csv", index=False)
curve_test.to_csv(RES / "05_reliability_test.csv", index=False)

final_metrics = {
    "final_model": {"key": FINAL_KEY, "label": str(FINAL_ROW.model),
                    "strategy": str(FINAL_ROW.strategy), "source": str(FINAL_ROW.source)},
    "frozen_operating_point": FROZEN,
    "selected_on": "validation only",
    "test": {
        "n": int(len(y_test)), "base_rate": float(y_test.mean()),
        "pr_auc": float(test_m["pr_auc"]), "roc_auc": float(test_m["roc_auc"]),
        "brier": float(test_m["brier"]), "log_loss": float(test_m["log_loss"]),
        "ece": float(ece_test),
        "precision": float(precision_score(y_test, flag_test)),
        "recall": float(recall_score(y_test, flag_test)),
        "f1": float(f1_score(y_test, flag_test)),
        "flag_rate": float(flag_test.mean()),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
        "pr_auc_lift": float(test_m["pr_auc"] / y_test.mean()),
    },
    "validation": {"pr_auc": float(val_m["pr_auc"]), "roc_auc": float(val_m["roc_auc"]),
                   "brier": float(val_m["brier"]), "ece": float(ece_val)},
    "cost": {
        "params": COSTS.__dict__,
        "total_at_frozen_threshold_inr": test_at_thr["total_cost_inr"],
        "total_at_0.5_threshold_inr": test_at_half["total_cost_inr"],
        "cost_of_using_default_threshold_inr": float(saving_vs_half),
        "no_model_baselines_inr": baselines,
        "saving_vs_best_no_model_inr": float(saving_vs_nomodel),
        "cost_per_order_inr": test_at_thr["cost_per_order_inr"],
        "three_tier": three_tier,
        "sensitivity_threshold_range": [float(sens.optimal_threshold.min()),
                                        float(sens.optimal_threshold.max())],
    },
    "three_tier_policy": tier_tbl.to_dict(orient="records"),
    "prevalence_shift": {
        "range_tested": [float(RATES.min()), float(RATES.max())],
        "published_range": [0.18, 0.35],
        "min_pr_auc_lift_in_published_range": float(lift_min),
        "beats_no_model_everywhere_in_range": never_worse,
        "worst_regret_inr_per_order": float(worst.regret_frozen),
        "worst_regret_at_prevalence": float(worst.prevalence),
        "worst_regret_with_prior_correction_inr": float(in_range.regret_corrected.max()),
        "ece_untreated_max": float(in_range.ece_raw.max()),
        "ece_corrected_max": float(in_range.ece_corrected.max()),
    },
    "bayes_ceiling_val": bench["bayes_ceiling_val"],
    "data_claim": (
        "Trained and evaluated on synthetic orders built on the real India Post "
        "pincode directory and calibrated against published Indian RTO statistics "
        "(enforced by test). No claim of validation on real merchant data is made."
    ),
}
with open(RES / "05_final_metrics.json", "w", encoding="utf-8") as fh:
    json.dump(final_metrics, fh, indent=2, default=float)

print(f"wrote models/final_model.joblib")
for f in ["05_final_metrics.json", "05_test_predictions.parquet",
          "05_prevalence_shift.csv", "05_exception_list.csv",
          "05_cost_sensitivity.csv", "05_reliability_test.csv"]:
    print(f"wrote reports/results/{f}")

In [ ]:
print(f"""
{'=' * 74}
FINAL RESULT -- {FINAL_ROW.model} ({FINAL_ROW.strategy})
{'=' * 74}

  TEST SET, {len(y_test):,} orders, opened once, base rate {y_test.mean():.3f}

    PR-AUC        {test_m['pr_auc']:.4f}   ({test_m['pr_auc'] / y_test.mean():.2f}x the no-skill floor)
    ROC-AUC       {test_m['roc_auc']:.4f}
    Brier         {test_m['brier']:.4f}    ECE {ece_test:.4f}
    precision     {precision_score(y_test, flag_test):.4f}
    recall        {recall_score(y_test, flag_test):.4f}
    F1            {f1_score(y_test, flag_test):.4f}

  AT THE COST MINIMUM (threshold {THRESHOLD:.3f}, chosen on validation)

    cost per order            Rs{test_at_thr['cost_per_order_inr']:.2f}
    vs best no-model policy   Rs{saving_vs_nomodel:,.0f} saved
    vs the 0.5 default        Rs{saving_vs_half:,.0f} saved
    threshold sensitivity     {sens.optimal_threshold.min():.3f} - {sens.optimal_threshold.max():.3f} across the assumed cost ranges

  THREE-TIER POLICY

    allow COD          {tier_tbl.share.iat[0]:.0%} of orders, {tier_tbl.actual_rto_rate.iat[0]:.1%} actual RTO
    charge a COD fee   {tier_tbl.share.iat[1]:.0%} of orders, {tier_tbl.actual_rto_rate.iat[1]:.1%} actual RTO
    disable COD        {tier_tbl.share.iat[2]:.0%} of orders, {tier_tbl.actual_rto_rate.iat[2]:.1%} actual RTO

  PREVALENCE SHIFT, 18% - 35%

    minimum PR-AUC lift       {lift_min:.2f}x
    beats no-model everywhere {never_worse}
    worst regret              Rs{worst.regret_frozen:.2f}/order at {worst.prevalence:.0%}
    with prior correction     Rs{in_range.regret_corrected.max():.2f}/order

  Synthetic data on a real pincode skeleton, calibrated to published statistics
  by assertion. No claim of validation on real merchant data.
""")

---

## What 05 established

| | |
|---|---|
| Selection | one model, on validation only |
| Test set | opened once, at frozen settings |
| Threshold | rupee cost minimum, with the cost of the 0.5 default stated |
| Calibration | Brier + reliability diagram + ECE, reported at the operating point |
| Prevalence shift | measured across 18–35%, with the regret and a one-line remedy |
| Exceptions | named and quantified, including the features `02` predicted would be weak |

Every figure is in `reports/figures/`, every metric in `reports/results/`. No number
in the README is unbacked by a file there.